In [ ]:
import numpy as np
import os
import re
import matplotlib.pyplot as plt
import fabio

from scipy.stats import binned_statistic_2d

In [ ]:
dir_coupled = r"C:\Users\j.bantol\Documents\Data\RSM\2026-06-29_gao_sto01.3\1_EIGERfull-2Dsnapshot_RSM-STO026s_34.9-37deg_0.01deg_3s_FrameFiles\frames"
dir_rocking = r"C:\Users\j.bantol\Documents\Data\RSM\2026-06-29_gao_sto01.3\2_EIGERfull-2Dsnapshot_rocking-STO026s_34.5-36deg_0.01deg_0.1s_FrameFiles\frames"

In [ ]:
%matplotlib widget
plt.close("all")

frames = sorted([f for f in os.listdir(dir_coupled) if f.endswith((".gfrm"))])
print(f"Found {len(frames)} frames")

# plot reference frame
ref_obj = fabio.open(os.path.join(dir_coupled, frames[0]))
ref_data = ref_obj.data
#ref_data = np.rot90(ref_obj.data, k=1)
print("pixel size:", ref_data.shape)

fig, ax = plt.subplots(figsize=(8, 5))
img = ax.imshow(ref_data, vmax=np.percentile(ref_data, 99.5), origin="lower", cmap="viridis")
plt.colorbar(img, ax=ax, label="Intensity / counts")

ax.set_title(f"Reference frame", fontsize="12")
ax.set_xlabel(f"pixel", fontsize="11")
ax.set_ylabel(f"pixel", fontsize="11")
plt.show()

In [ ]:
# instrument parameters
wavelength = 0.71076      # Å, Mo Kα
pixel_size = 0.075        # mm (75 µm)
detector_distance = 310   # mm
center_col = 494.03       # px, beam center column
center_row = 226.5        # px, beam center row    

In [ ]:
# for each frame, convert center pixel to 2theta
def pixel_to_2theta(pixel, center_pixel, two_theta_center, detector_distance, pixel_size): 
    offset_mm = (pixel - center_pixel) * pixel_size
    delta_2theta = np.rad2deg(np.arctan(offset_mm / detector_distance))    # opposite is offset_mm
    
    return two_theta_center + delta_2theta

# convert omega, 2theta to Qy, Qz
def angles_to_Q(omega, two_theta, wavelength, chi=18.4349):     # chi=tan-1(1/6 / 1/2) for angle between (026) and (001)
    k = 2 * np.pi / wavelength
    alpha_i = np.deg2rad(omega)
    alpha_f = np.deg2rad(two_theta - omega)
    chi_r   = np.deg2rad(chi)

    # Q in diffractometer frame (before tilt)
    Q_perp = k * (np.sin(alpha_i) + np.sin(alpha_f))  # along diffractometer z
    Q_par  = k * (np.cos(alpha_i) - np.cos(alpha_f))  # along diffractometer y

    # rotate by chi to get into crystal frame (026s geometry, phi=-90)
    Qy =  Q_perp * np.sin(chi_r) + Q_par * np.cos(chi_r)
    Qz =  Q_perp * np.cos(chi_r) - Q_par * np.sin(chi_r)

    return Qy, Qz

In [ ]:
# for one frame

## scan parameters
### coupled omega-2theta scan
omega = 34.9
count_time_c = 3
two_theta_center = 2 * omega
chi = 18.4349

In [ ]:
frames = sorted([f for f in os.listdir(dir_coupled) if f.endswith((".gfrm"))])
obj = fabio.open(os.path.join(dir_coupled, frames[0]))
data = obj.data.astype(float) / count_time_c

nrows = data.shape[0]
ncols = data.shape[1]

Qy_frame = []
Qz_frame = []
I_frame = []

for col in range(ncols):
    for row in range(nrows):
        intensity = data[row, col]
        if intensity <= 0:
            continue
            
        # cols to 2theta to Qy and Qz via chi rotation
        tt_pixel = pixel_to_2theta(col, center_col, two_theta_center, detector_distance, pixel_size)
        Qy_pix, Qz_pix = angles_to_Q(omega, tt_pixel, wavelength)
        
        # rows to rows with offset
        offset_row = (row - center_row) * pixel_size
        delta_2theta_row = np.rad2deg(np.arctan(offset_row / detector_distance))
        delta_Qy = (2 * np.pi / wavelength) * np.deg2rad(delta_2theta_row)
        
        Qy_frame.append(Qy_pix + delta_Qy)
        #Qy_frame.append(Qy_pix)
        Qz_frame.append(Qz_pix)
        I_frame.append(intensity)

Qy_frame = np.array(Qy_frame)
Qz_frame = np.array(Qz_frame)
I_frame  = np.array(I_frame)

In [ ]:
I_grid, Qy_edges, Qz_edges, _ = binned_statistic_2d(Qy_frame, Qz_frame, I_frame, statistic="mean", bins=[300, 300],
                                                    range=[[Qy_frame.min(), Qy_frame.max()], [Qz_frame.min(), Qz_frame.max()]])
I_grid = np.nan_to_num(I_grid, nan=0)
Qy_centers = (Qy_edges[:-1] + Qy_edges[1:]) / 2
Qz_centers = (Qz_edges[:-1] + Qz_edges[1:]) / 2

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
im = ax.pcolormesh(Qy_centers, Qz_centers, np.log10(I_grid.T + 1),
                   cmap="viridis",
                   vmin=np.nanpercentile(np.log10(I_grid[I_grid>0]+1), 50),
                   vmax=np.nanpercentile(np.log10(I_grid+1), 99.99))
plt.colorbar(im, ax=ax, label='log₁₀(Intensity / cps)')
ax.set_xlabel("Qy / Å⁻¹")
ax.set_ylabel("Qz / Å⁻¹")
ax.set_title(f"Single frame RSM: ω={omega:.2f}°")
plt.show()

In [ ]:
obj = fabio.open(os.path.join(dir_coupled, frames[0]))
data = obj.data

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(data.sum(axis=0))  # vs col
axes[0].set_xlabel("col")
axes[0].set_title("horizontal profile")

axes[1].plot(data.sum(axis=1))  # vs row
axes[1].set_xlabel("row")
axes[1].set_title("vertical profile")

plt.tight_layout()
plt.show()

In [ ]:
# for multiple frames

## scan parameters
### coupled omega-2theta scan
omega_start_c = 34.9
omega_step_c = 0.01
count_time_c = 3.0

### rocking scan
omega_start_r = 34.5
omega_step_r = 0.01
count_time_r = 0.1
two_theta_fixed_r = 70.1144  # fixed, 2theta = 2 * omega

In [ ]:
all_Qy = []
all_Qz = []
all_I = []
all_sctype = []

def process_data(directory, omega_start, omega_step, count_time, scan_type="coupled", two_theta_fixed=None):
    frames = sorted([f for f in os.listdir(directory) if f.endswith((".gfrm"))])
    print(f"For {scan_type} scan: {len(frames)} frames")
    
    for i, f in enumerate(frames):
        obj = fabio.open(os.path.join(directory, f))
        data = obj.data.astype(float) / count_time  # normalize to cps
        #data = np.rot90(orig_data, k=1)
    
        omega = omega_start + i * omega_step
    
        if scan_type == "coupled":
                two_theta_center = 2 * omega
        else:
                two_theta_center = two_theta_fixed
        
        # rows are perpendicular to the diffraction plane (Debye-Scherrer arc direction)
        # they don't map onto a different in-plane Qy/Qz, so integrate them out per column, instead of converting row pixel position into a spurious Q shift
        intensity_per_col = data.sum(axis=0)   # shape (ncols,)
        
        # cols to 2theta to Qy and Qz via chi rotation, vectorized over the whole column axis
        ncols = data.shape[1]
        cols = np.arange(ncols)
        tt_pixel = pixel_to_2theta(cols, center_col, two_theta_center, detector_distance, pixel_size)
        Qy_col, Qz_col = angles_to_Q(omega, tt_pixel, wavelength)     # shape (ncols,)
        
        # rows to rows with offset to delta Qy, vectorized over the whole row axis
        #rows = np.arange(nrows)
        #offset_row = (rows - center_row) * pixel_size
        #delta_Qy = (2 * np.pi / wavelength) * np.arctan(offset_row / detector_distance)    # shape of nrows

        #Qy_grid = Qy_col[None, :] + delta_Qy[:, None]       # shape (nrows, ncols), matches data
        #Qz_grid = np.broadcast_to(Qz_col[None, :], data.shape)
        
        #mask = data > 0
        mask = intensity_per_col > 0
        all_Qy.append(Qy_col[mask])
        all_Qz.append(Qz_col[mask])
        all_I.append(intensity_per_col[mask])
        all_sctype.append(np.full(mask.sum(), scan_type))
    
    global Qy, Qz, I, sctype
    Qy = np.concatenate(all_Qy)
    Qz = np.concatenate(all_Qz)
    I = np.concatenate(all_I)
    sctype = np.concatenate(all_sctype)

In [ ]:
# process scans
process_data(dir_coupled, omega_start_c, omega_step_c, count_time_c, scan_type="coupled")
process_data(dir_rocking, omega_start_r, omega_step_r, count_time_r, scan_type="rocking", two_theta_fixed=two_theta_fixed_r)

In [ ]:
print(f"Total points: {len(Qy)}")
print(f"Qy range: {Qy.min():.4f} to {Qy.max():.4f} Å⁻¹")
print(f"Qz range: {Qz.min():.4f} to {Qz.max():.4f} Å⁻¹")

In [ ]:
grids = {}
for label in ["coupled", "rocking"]:
    mask = sctype == label
    Qy_s, Qz_s, I_s = Qy[mask], Qz[mask], I[mask]
    
    I_grid, Qy_edges, Qz_edges, _ = binned_statistic_2d(Qy_s, Qz_s, I_s, statistic="mean", bins=[200, 200],
                                                        range=[[Qy_s.min(), Qy_s.max()], [Qz_s.min(), Qz_s.max()]])
    I_grid = np.nan_to_num(I_grid, nan=0)
    Qy_centers = (Qy_edges[:-1] + Qy_edges[1:]) / 2
    Qz_centers = (Qz_edges[:-1] + Qz_edges[1:]) / 2

    grids[label] = (I_grid, Qy_centers, Qz_centers)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, label in zip(axes, ["coupled", "rocking"]):
    I_grid, Qy_centers, Qz_centers = grids[label]
    im = ax.pcolormesh(Qy_centers, Qz_centers, np.log10(I_grid.T + 1),
                       cmap="hot",
                       vmin=np.nanpercentile(np.log10(I_grid[I_grid>0]+1), 70),
                       vmax=np.nanpercentile(np.log10(I_grid+1), 99.99))
    plt.colorbar(im, ax=ax, label="log₁₀(Intensity / cps)")
    ax.set_xlabel("Qy / Å⁻¹")
    ax.set_ylabel("Qz / Å⁻¹")
    ax.set_title(f"{label} scan RSM at STO(026)")

plt.tight_layout()
plt.show()

In [ ]:
for label, (I_grid, _, _) in grids.items():
    print(f"{label}: valid intensity sum = {np.nansum(I_grid):.1f}, NaN count = {np.sum(np.isnan(I_grid))}")
print("Qy sample:", Qy[:5])
print("Qz sample:", Qz[:5])
print("I sample:", I[:5])

In [ ]:
from scipy.ndimage import maximum_filter

def find_local_peaks(I_grid, Qy_centers, Qz_centers, min_distance=5, top_n=2):
    """Return the top_n distinct local intensity maxima (film/substrate),
    instead of a single global argmax which can pick a different physical
    peak in each scan if their relative brightness differs.
    I_grid axis 0 is Qy, axis 1 is Qz (binned_statistic_2d(Qy, Qz, ...) convention)."""
    footprint = np.ones((min_distance, min_distance))
    local_max = (I_grid == maximum_filter(I_grid, footprint=footprint)) & (I_grid > 0)
    iQy_all, iQz_all = np.nonzero(local_max)
    order = np.argsort(I_grid[iQy_all, iQz_all])[::-1]

    peaks = []
    for idx in order:
        iQy, iQz = iQy_all[idx], iQz_all[idx]
        if any(abs(iQy - pQy) < min_distance and abs(iQz - pQz) < min_distance for pQy, pQz, _ in peaks):
            continue
        peaks.append((iQy, iQz, I_grid[iQy, iQz]))
        if len(peaks) == top_n:
            break

    return [(Qy_centers[iQy], Qz_centers[iQz], intensity) for iQy, iQz, intensity in peaks]

for label in ['coupled', 'rocking']:
    I_grid, Qy_centers, Qz_centers = grids[label]
    peaks = find_local_peaks(I_grid, Qy_centers, Qz_centers)
    print(f"{label} scan:")
    for Qy_pk, Qz_pk, intensity in peaks:
        print(f"  Qy={Qy_pk:.4f}, Qz={Qz_pk:.4f}  (I={intensity:.1f} cps)")

In [ ]:
# overlay the substrate location (from find_local_peaks) and the PREDICTED film
# location (substrate + the coupled-scan-confirmed offset) directly on the actual
# zoomed maps, so we can see exactly where the prediction falls relative to
# whatever is visually there - rather than trusting only a 1D line cut through it
substrate_by_scan = {'coupled': (3.2151, 9.6349), 'rocking': (3.2097, 9.6352)}
film_offset = np.array([3.2245, 9.6869]) - np.array([3.2151, 9.6349])

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, label in zip(axes, ['coupled', 'rocking']):
    I_grid, Qy_centers, Qz_centers = grids[label]
    im = ax.pcolormesh(Qy_centers, Qz_centers, np.log10(I_grid.T + 1),
                        cmap='hot', vmin=2.0, vmax=3.6)
    plt.colorbar(im, ax=ax, label='log₁₀(Intensity / cps)')

    sub_Qy, sub_Qz = substrate_by_scan[label]
    film_Qy, film_Qz = sub_Qy + film_offset[0], sub_Qz + film_offset[1]

    ax.plot(sub_Qy, sub_Qz, marker='+', color='cyan', markersize=18, markeredgewidth=2, label='substrate (measured)')
    ax.plot(film_Qy, film_Qz, marker='x', color='lime', markersize=18, markeredgewidth=2, label='predicted film location')

    ax.set_xlim(3.05, 3.35)
    ax.set_ylim(9.55, 9.75)
    ax.set_xlabel("Qy / \u00c5\u207b\u00b9")
    ax.set_ylabel("Qz / \u00c5\u207b\u00b9")
    ax.set_title(f"{label} scan (zoomed)")
    ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
def dump_frame_header(directory, frame_indices=(0, None, -1)):
    frames = sorted([f for f in os.listdir(directory) if f.endswith(".gfrm")])
    n = len(frames)
    indices = [i if i is not None else n // 2 for i in frame_indices]
    for i in indices:
        obj = fabio.open(os.path.join(directory, frames[i]))
        print(f"--- frame index {i}: {frames[i]} ---")
        header = obj.header
        # print any header keys that look angle/goniometer related
        for k, v in header.items():
            if re.search(r"ANGLE|OMEGA|THETA|PHI|CHI|2THETA|GONIO", k, re.IGNORECASE):
                print(f"  {k}: {v}")
        print()

print("=== COUPLED scan header check ===")
dump_frame_header(dir_coupled)

print("=== ROCKING scan header check ===")
dump_frame_header(dir_rocking)

In [ ]:
def compare_scans(labelA, labelB, n_bins=150):
    """Quantitative check of whether two processed scans agree in their
    overlapping Q region - the actual test of whether the (omega,2theta)->
    (Qy,Qz) conversion makes different scan geometries equivalent, rather
    than assuming it because the substrate peak alone lines up."""
    maskA = sctype == labelA
    maskB = sctype == labelB
    Qy_lo = max(Qy[maskA].min(), Qy[maskB].min())
    Qy_hi = min(Qy[maskA].max(), Qy[maskB].max())
    Qz_lo = max(Qz[maskA].min(), Qz[maskB].min())
    Qz_hi = min(Qz[maskA].max(), Qz[maskB].max())

    if Qy_lo >= Qy_hi or Qz_lo >= Qz_hi:
        print(f"{labelA} vs {labelB}: no Q-space overlap")
        return None

    gridA, _, _, _ = binned_statistic_2d(Qy[maskA], Qz[maskA], I[maskA], statistic='mean',
                                          bins=[n_bins, n_bins], range=[[Qy_lo, Qy_hi], [Qz_lo, Qz_hi]])
    gridB, _, _, _ = binned_statistic_2d(Qy[maskB], Qz[maskB], I[maskB], statistic='mean',
                                          bins=[n_bins, n_bins], range=[[Qy_lo, Qy_hi], [Qz_lo, Qz_hi]])

    valid = ~np.isnan(gridA) & ~np.isnan(gridB) & (gridA > 0) & (gridB > 0)
    if valid.sum() < 10:
        print(f"{labelA} vs {labelB}: overlap region has almost no shared signal ({valid.sum()} bins)")
        return None

    corr = np.corrcoef(np.log10(gridA[valid]), np.log10(gridB[valid]))[0, 1]
    ratio = np.median(gridA[valid] / gridB[valid])
    print(f"{labelA} vs {labelB}: {valid.sum()} overlapping bins, log-log correlation={corr:.3f}, median ratio (A/B)={ratio:.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    axes[0].scatter(np.log10(gridB[valid]), np.log10(gridA[valid]), s=3, alpha=0.3)
    lo = min(axes[0].get_xlim()[0], axes[0].get_ylim()[0])
    hi = max(axes[0].get_xlim()[1], axes[0].get_ylim()[1])
    axes[0].plot([lo, hi], [lo, hi], 'r--', label='y = x')
    axes[0].set_xlabel(f"log10 I ({labelB})")
    axes[0].set_ylabel(f"log10 I ({labelA})")
    axes[0].legend()
    axes[0].set_title(f"{labelA} vs {labelB}  (r={corr:.3f})")

    Qy_edges_cmp = np.linspace(Qy_lo, Qy_hi, n_bins + 1)
    Qz_edges_cmp = np.linspace(Qz_lo, Qz_hi, n_bins + 1)
    ratio_map = np.where(valid, np.log10(gridA / gridB), np.nan)
    im = axes[1].pcolormesh(Qy_edges_cmp, Qz_edges_cmp, ratio_map.T, cmap='coolwarm', vmin=-1, vmax=1)
    plt.colorbar(im, ax=axes[1], label=f"log10({labelA}/{labelB})")
    axes[1].set_xlabel("Qy / \u00c5\u207b\u00b9")
    axes[1].set_ylabel("Qz / \u00c5\u207b\u00b9")
    axes[1].set_title("intensity ratio across the overlap")

    plt.tight_layout()
    plt.show()
    return corr, ratio

# check every pair of labels currently present in sctype - add more process_data()
# calls with new labels (e.g. more rocking scans) and this keeps working unchanged
labels_present = sorted(set(sctype))
for i in range(len(labels_present)):
    for j in range(i + 1, len(labels_present)):
        compare_scans(labels_present[i], labels_present[j])


In [ ]:
def find_frame_and_column(target_Qy, target_Qz, directory, omega_start, omega_step,
                          two_theta_fixed, wavelength, chi=18.4349):
    """Find which raw frame + column in a fixed-2theta scan is geometrically
    closest to a target (Qy, Qz), so the raw data can be inspected directly
    instead of trusting only the binned/aggregated histogram."""
    frames = sorted([f for f in os.listdir(directory) if f.endswith(".gfrm")])
    sample = fabio.open(os.path.join(directory, frames[0]))
    ncols = sample.data.shape[1]
    cols = np.arange(ncols)

    best = None
    for fi, f in enumerate(frames):
        omega = omega_start + fi * omega_step
        tt = pixel_to_2theta(cols, center_col, two_theta_fixed, detector_distance, pixel_size)
        Qy_line, Qz_line = angles_to_Q(omega, tt, wavelength, chi)
        d2 = (Qy_line - target_Qy) ** 2 + (Qz_line - target_Qz) ** 2
        ci = np.argmin(d2)
        if best is None or d2[ci] < best[0]:
            best = (d2[ci], fi, f, cols[ci], omega, Qy_line[ci], Qz_line[ci])

    dist2, fi, fname, col, omega, Qy_hit, Qz_hit = best
    print(f"closest match: frame {fi} ({fname}), omega={omega:.4f}, column={col}, "
          f"achieves Qy={Qy_hit:.4f}, Qz={Qz_hit:.4f}  "
          f"(target Qy={target_Qy:.4f}, Qz={target_Qz:.4f}, dist={np.sqrt(dist2):.5f})")
    return fi, fname, col

# use the coupled scan's own confirmed second peak (film candidate) as the target
coupled_peaks = find_local_peaks(*grids['coupled'])
film_Qy, film_Qz = coupled_peaks[1][0], coupled_peaks[1][1]
print(f"target film location (from coupled scan): Qy={film_Qy:.4f}, Qz={film_Qz:.4f}\n")

fi, fname, col = find_frame_and_column(film_Qy, film_Qz, dir_rocking, omega_start_r, omega_step_r,
                                        two_theta_fixed_r, wavelength)

# plot that EXACT raw frame's column profile directly - no binning/aggregation involved
obj = fabio.open(os.path.join(dir_rocking, fname))
data = obj.data.astype(float) / count_time_r
col_profile = data.sum(axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(col_profile)
ax.axvline(col, color='red', linestyle='--', label=f'predicted film column ({col})')
ax.set_xlabel("column")
ax.set_ylabel("row-summed intensity / cps")
ax.set_title(f"raw column profile: {fname}  (rocking, \u03c9={omega_start_r + fi*omega_step_r:.3f}\u00b0)")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
def decompose_strain_tilt(substrate_Q, film_Q):
    """Decompose a film-peak offset from substrate into:
    - a radial component (along the substrate's own Q-vector direction) - the
      strain: fractional change in d-spacing along this reflection
    - a tangential component (perpendicular to that) - the tilt: an angular
      misorientation between film and substrate lattice planes
    This is only a valid decomposition if the film peak truly lies in the same
    2D (Qy,Qz) plane assumed by the conversion (rows integrated away assuming
    no relevant out-of-plane component) - a large tilt can violate that."""
    substrate_Q = np.asarray(substrate_Q, dtype=float)
    film_Q = np.asarray(film_Q, dtype=float)
    dQ = film_Q - substrate_Q

    Q_mag = np.linalg.norm(substrate_Q)
    Q_hat = substrate_Q / Q_mag
    perp_hat = np.array([-Q_hat[1], Q_hat[0]])

    dQ_radial = np.dot(dQ, Q_hat)
    dQ_tangential = np.dot(dQ, perp_hat)

    strain_pct = -dQ_radial / Q_mag * 100
    tilt_deg = np.degrees(dQ_tangential / Q_mag)
    return dict(dQ_radial=dQ_radial, dQ_tangential=dQ_tangential,
                strain_pct=strain_pct, tilt_deg=tilt_deg)


results = {}
for label in ['coupled', 'rocking']:
    I_grid, Qy_centers, Qz_centers = grids[label]
    peaks = find_local_peaks(I_grid, Qy_centers, Qz_centers)
    (sub_Qy, sub_Qz, sub_I), (film_Qy, film_Qz, film_I) = peaks

    result = decompose_strain_tilt((sub_Qy, sub_Qz), (film_Qy, film_Qz))
    results[label] = result

    print(f"{label} scan  (substrate Qy={sub_Qy:.4f}, Qz={sub_Qz:.4f}  |  film Qy={film_Qy:.4f}, Qz={film_Qz:.4f}):")
    print(f"  radial (strain-sensitive)   \u0394Q = {result['dQ_radial']:+.5f} \u00c5\u207b\u00b9  ->  strain = {result['strain_pct']:+.4f} %")
    print(f"  tangential (tilt-sensitive) \u0394Q = {result['dQ_tangential']:+.5f} \u00c5\u207b\u00b9  ->  tilt   = {result['tilt_deg']:+.4f}\u00b0")
    print()

# sanity check: the radial (strain) component should roughly agree in SIGN between
# the two scans if this is genuinely one film peak with coupled cleanly measuring
# strain and rocking mostly measuring tilt. A sign flip here means either the tilt
# is large enough that the film peak sits meaningfully outside the flat (Qy,Qz)
# plane this code assumes (rows integrated away), or these aren't the same feature.
if set(results) == {'coupled', 'rocking'}:
    same_sign = np.sign(results['coupled']['strain_pct']) == np.sign(results['rocking']['strain_pct'])
    print(f"strain sign agreement between scans: {'OK' if same_sign else 'MISMATCH - treat these numbers with caution, see note above'}")


In [ ]:
# |Q| is invariant under a pure tilt (rotation) of the crystal - only real strain
# changes |Q|. So if the "film peak" found in two different scans is genuinely the
# SAME reflection (just viewed with different tilt/strain sensitivity), |Q_film|
# must agree between them. If it doesn't - by more than the bin resolution - they
# are most likely two different physical reflections, not one peak seen two ways.
Q_sub_mag = {}
Q_film_mag = {}
for label in ['coupled', 'rocking']:
    I_grid, Qy_centers, Qz_centers = grids[label]
    (sub_Qy, sub_Qz, _), (film_Qy, film_Qz, _) = find_local_peaks(I_grid, Qy_centers, Qz_centers)
    Q_sub_mag[label] = np.hypot(sub_Qy, sub_Qz)
    Q_film_mag[label] = np.hypot(film_Qy, film_Qz)
    print(f"{label}: |Q_substrate| = {Q_sub_mag[label]:.4f}, |Q_film| = {Q_film_mag[label]:.4f}  "
          f"(strain vs substrate: {100*(Q_film_mag[label]-Q_sub_mag[label])/Q_sub_mag[label]:+.3f} %)")

if set(Q_film_mag) == {'coupled', 'rocking'}:
    d_Qmag = Q_film_mag['coupled'] - Q_film_mag['rocking']
    print(f"\n|Q_film| difference between scans: {d_Qmag:+.4f} \u00c5\u207b\u00b9 "
          f"({100*abs(d_Qmag)/np.mean(list(Q_sub_mag.values())):.2f}% of |Q_substrate|)")
    print("If this is much larger than the grid's bin resolution, the two 'film' peaks")
    print("are likely different physical reflections, not the same peak seen two ways -")
    print("tilt alone cannot produce a |Q| mismatch this size.")
